# Free Tokens: Speculative Decoding, from the Math to vLLM

A 7B model in bf16 is ~14 GB of weights, and autoregressive decoding reads *all of
them* to emit *one token*. That is roughly one FLOP per byte moved — an H100 needs
hundreds of FLOPs per byte before compute is the limit, so during decode the tensor
cores are almost entirely idle. The model isn't slow because it computes too much; it's
slow because it re-reads itself constantly.

Speculative decoding attacks the read itself: use something cheap to **propose** several
tokens, then let the target model **verify all of them in one forward pass** — one weight
read, several tokens of useful work. A rejection-sampling rule makes the output
*provably identical in distribution* to decoding with the target model alone.

This notebook covers the full arc:

1. the roofline argument for why decode is memory-bound;
2. the draft-and-verify idea, and why verification is almost free;
3. the rejection-sampling math, with the proof that quality is untouched;
4. a **from-scratch implementation** you can run on two GPT-2 models;
5. the modern proposal zoo — draft models, n-gram lookup, Medusa, EAGLE;
6. a **production benchmark harness** for vLLM (baseline vs. n-gram vs. EAGLE-3), with
   measured H100 numbers — including the counterintuitive result that n-gram speculation
   is *slower* than no speculation on this hardware.

## 1 · Why decode is memory-bound

Per generated token, a dense 7B model does ≈ $2 \times 7\mathrm{B} = 14$ GFLOP while
reading ≈ 14 GB of weights (bf16): **arithmetic intensity ≈ 1 FLOP/byte**. An H100 SXM
delivers ~990 TFLOP/s (bf16) against ~3.35 TB/s of HBM bandwidth — it needs ≈ **300
FLOPs per byte** to be compute-limited. At intensity 1, the achievable ceiling is set
purely by bandwidth:

$$\text{max tok/s} \approx \frac{3350\ \text{GB/s}}{14\ \text{GB}} \approx 239$$

and real engines land at 60–70% of that after KV-cache reads and scheduling overhead.
The key observation: a forward pass over $K{+}1$ tokens costs *almost the same* as over
one token — the weight read dominates and is identical; only the (tiny, at small $K$)
activation work scales. Verification is nearly free; proposing is the hard part.

## 2 · The algorithm

```
loop until enough tokens:
    PROPOSE  cheap source suggests d₁ … d_K            (K cheap steps)
    VERIFY   target runs ONE forward over all of them  (1 weight read)
    ACCEPT   walk left→right, keep each dᵢ with prob min(1, p(dᵢ)/q(dᵢ))
    PATCH    on first rejection, resample from norm(max(0, p − q)) and stop
    BONUS    if all K survived, the verify pass already gives token K+1 free
```

Best case: $K{+}1$ tokens per weight read. Worst case: 1 token — never slower in
*tokens per pass*, though proposal overhead can still lose wall-clock time (§7 shows a
real example).

## 3 · Why the output distribution is exactly preserved

Let $q$ be the draft distribution and $p$ the target distribution at some position. A
drafted token $x \sim q$ is accepted with probability $\min(1, p(x)/q(x))$; on
rejection we sample from the *residual* $p'(x) = \max(0, p(x) - q(x)) / Z$.

The probability that this procedure outputs token $x$:

$$\underbrace{q(x)\min\!\left(1, \tfrac{p(x)}{q(x)}\right)}_{\text{accepted}}
+ \underbrace{\Big(1 - \sum_y \min(p(y), q(y))\Big)}_{P(\text{reject})} \cdot
\underbrace{\frac{\max(0, p(x) - q(x))}{\sum_y \max(0, p(y) - q(y))}}_{p'(x)}$$

The first term is $\min(p(x), q(x))$. Since
$\sum_y \max(0, p(y)-q(y)) = 1 - \sum_y \min(p(y), q(y))$, the rejection factor and
the residual's normalizer cancel, leaving

$$\min(p(x), q(x)) + \max(0, p(x) - q(x)) = p(x).$$

Every emitted token is distributed exactly as if the target model had produced it —
regardless of how bad the draft is. A bad draft costs *speed*, never *quality*
([Leviathan et al., 2023](https://arxiv.org/abs/2211.17192);
[Chen et al., 2023](https://arxiv.org/abs/2302.01318)).

## 4 · From scratch

The implementation mirrors the algorithm box: `propose` runs the draft model $K$ steps,
`accept_or_patch` applies the rejection rule, and the driver loop stitches iterations
together. Kept deliberately cache-free for readability — production versions reuse KV
caches for both models, which changes constants but not logic.

In [ ]:
import torch
import torch.nn.functional as F

@torch.inference_mode()
def propose(draft_lm, seq, K, temp):
    """Draft K tokens autoregressively. Returns extended seq + the K draft distributions."""
    dists = []
    for _ in range(K):
        logits = draft_lm(seq).logits[:, -1]
        dist = F.softmax(logits / max(temp, 1e-6), dim=-1)
        nxt = torch.multinomial(dist, 1)
        seq = torch.cat([seq, nxt], dim=1)
        dists.append(dist)
    return seq, torch.stack(dists, dim=1)          # (B, K, vocab)

@torch.inference_mode()
def speculative_generate(target_lm, draft_lm, prompt_ids, n_new, K=4, temp=1.0):
    """Sample n_new tokens from target_lm's distribution, accelerated by draft_lm.
    Returns (sequence, accepted_count, proposal_count) for acceptance-rate stats."""
    seq = prompt_ids.clone()
    goal = seq.shape[1] + n_new
    n_accepted = n_proposed = 0

    while seq.shape[1] < goal:
        start = seq.shape[1]
        drafted, q = propose(draft_lm, seq, K, temp)

        # one target pass scores every drafted position (plus the bonus position)
        logits = target_lm(drafted).logits[:, start - 1: start + K]
        p = F.softmax(logits / max(temp, 1e-6), dim=-1)        # (B, K+1, vocab)

        kept = 0
        for t in range(K):
            tok = drafted[0, start + t]
            n_proposed += 1
            if torch.rand(()) < (p[0, t, tok] / q[0, t, tok]).clamp(max=1.0):
                kept += 1
                n_accepted += 1
            else:
                residual = (p[0, t] - q[0, t]).clamp(min=0)
                patch = torch.multinomial(residual / residual.sum(), 1)
                seq = torch.cat([seq, drafted[:, start:start + kept], patch[None]], dim=1)
                break
        else:
            bonus = torch.multinomial(p[0, K], 1)
            seq = torch.cat([drafted, bonus[None]], dim=1)

    return seq[:, :goal], n_accepted, n_proposed

### 4.1 · Watch it run

Two off-the-shelf models with the same tokenizer: GPT-2 (124M) drafts for GPT-2-large
(774M). Weak pairing by modern standards — which makes the acceptance rate an honest,
interesting number rather than a rigged demo.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

DEV = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained("gpt2")
draft = AutoModelForCausalLM.from_pretrained("gpt2").to(DEV).eval()
target = AutoModelForCausalLM.from_pretrained("gpt2-large").to(DEV).eval()

prompt = tok("The main reason data centers are built near rivers is", return_tensors="pt").input_ids.to(DEV)
torch.manual_seed(0)

t0 = time.perf_counter()
out, acc, prop = speculative_generate(target, draft, prompt, n_new=120, K=4, temp=0.8)
spec_s = time.perf_counter() - t0

torch.manual_seed(0)
t0 = time.perf_counter()
plain = target.generate(prompt, max_new_tokens=120, do_sample=True, temperature=0.8,
                        pad_token_id=tok.eos_token_id)
plain_s = time.perf_counter() - t0

print(tok.decode(out[0], skip_special_tokens=True), "\n")
print(f"acceptance rate : {acc}/{prop} = {acc / prop:.0%}")
print(f"speculative     : {120 / spec_s:.1f} tok/s   (no KV cache — see note above)")
print(f"plain generate  : {120 / plain_s:.1f} tok/s   (HF generate, cached)")

## 5 · The proposal zoo

Every method keeps the same verify step; they differ in where drafts come from.

| Method | Proposer | Extra memory | Needs training? | Sweet spot |
|---|---|---|---|---|
| **Draft model** | a small LM from the same family | ~1 GB (0.5B) | no | big targets (≥13B) with a good small sibling |
| **N-gram / prompt lookup** | match recent text against the prompt | ~0 | no | summarization, RAG, code edits — outputs that copy inputs |
| **Medusa** ([Cai et al., 2024](https://arxiv.org/abs/2401.10774)) | extra decoding heads on the target | <5% | yes (heads) | serving stacks that can fine-tune once |
| **EAGLE** ([Li et al., 2024](https://arxiv.org/abs/2401.15077)) | one-layer head that extrapolates the target's *feature* stream | <5% | yes (head) | best acceptance rates; current default |
| **EAGLE-3** ([Li et al., 2025](https://arxiv.org/abs/2503.01840)) | multi-layer feature fusion + training-time test | <5% | yes (head) | state of the art in vLLM |

The economics: speedup ≈ (accepted tokens per verify) ÷ (1 + proposal cost relative to a
target pass). Draft models propose well but cost real passes; n-gram is free but
proposes poorly on novel text; EAGLE proposes well *and* cheaply — which is why it wins
below.

## 6 · Production: benchmarking vLLM speculation

vLLM turns speculation on with a single `--speculative-config` flag. The harness below
starts a fresh server per configuration (clean state), sends a fixed prompt set through
the OpenAI-compatible API, and reports per-prompt and aggregate throughput.

Configurations under test on Qwen2.5-7B-Instruct:

- **baseline** — no speculation
- **ngram** — prompt-lookup drafting, $K=5$
- **eagle3** — a community EAGLE-3 head for this target, $K=3$

In [ ]:
import json as jsonlib
import os
import subprocess
import time

import requests

TARGET = "Qwen/Qwen2.5-7B-Instruct"
API = "http://127.0.0.1:8000"

SPEC_CONFIGS = {
    "baseline": None,
    "ngram": {"method": "ngram", "num_speculative_tokens": 5, "prompt_lookup_max": 4},
    "eagle3": {"method": "eagle3", "num_speculative_tokens": 3,
               "model": "ruipeterpan/Qwen2.5-7B-Instruct_EAGLE3_UltraChat",
               "draft_tensor_parallel_size": 1},
}

# Ten prompts across registers: expository, code, procedural, historical, technical.
BENCH_PROMPTS = [
    "Walk through how a garbage collector with generational heaps decides what to promote, what to sweep, and when to compact.",
    "Write a Rust function that parses an RFC 3339 timestamp without external crates, including error handling for malformed input.",
    "Explain how ocean currents redistribute heat around the planet, and what happens to regional climates if the AMOC weakens.",
    "Compare optimistic and pessimistic database locking. When does each win, and what failure modes should an engineer expect?",
    "Give step-by-step instructions for fermenting sourdough starter from scratch, including the schedule for the first ten days.",
    "Describe the full lifecycle of an HTTP request through a modern CDN, from DNS resolution to cache hit or origin fetch.",
    "Summarize the causes and consequences of the Bretton Woods agreement and why the system eventually collapsed in 1971.",
    "Write a Python asyncio worker pool with graceful shutdown, bounded concurrency, and per-task timeout, with type hints.",
    "Explain how mRNA vaccines work at the cellular level, from lipid nanoparticle uptake to antigen presentation.",
    "What is Byzantine fault tolerance? Explain the problem, the 3f+1 bound, and how PBFT reaches agreement.",
]

def launch_server(spec):
    os.system("pkill -f 'vllm' 2>/dev/null")
    time.sleep(4)
    cmd = ["vllm", "serve", TARGET, "--dtype", "bfloat16", "--seed", "0",
           "--max-model-len", "4096", "--gpu-memory-utilization", "0.85",
           "--port", "8000"]
    if spec:
        cmd += ["--speculative-config", jsonlib.dumps(spec)]
    proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    deadline = time.time() + 420
    while time.time() < deadline:
        try:
            if requests.get(f"{API}/health", timeout=2).ok:
                return proc
        except requests.RequestException:
            time.sleep(3)
    proc.kill()
    raise RuntimeError("vLLM server did not become healthy")

def ask(prompt, max_tokens=256):
    t0 = time.perf_counter()
    r = requests.post(f"{API}/v1/chat/completions", timeout=300, json={
        "model": TARGET, "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens, "temperature": 0})
    dt = time.perf_counter() - t0
    n = r.json()["usage"]["completion_tokens"]
    return n, dt

def bench_config(name, spec, warmup=3):
    proc = launch_server(spec)
    try:
        for p in BENCH_PROMPTS[:warmup]:
            ask(p, max_tokens=32)
        per_prompt, tokens, seconds = [], 0, 0.0
        for i, p in enumerate(BENCH_PROMPTS):
            n, dt = ask(p)
            per_prompt.append(n / dt)
            tokens += n
            seconds += dt
            print(f"  [{i + 1:>2}/10] {n:>3} tok  {dt:>6.2f}s  {n / dt:>7.1f} tok/s")
        return {"config": name, "tok_s": tokens / seconds,
                "ms_per_tok": seconds / tokens * 1000, "per_prompt": per_prompt}
    finally:
        proc.kill()
        os.system("pkill -f 'vllm' 2>/dev/null")
        time.sleep(5)

# Uncomment to run live (each config reloads the model: ~10 min total on an H100):
# live = {name: bench_config(name, spec) for name, spec in SPEC_CONFIGS.items()}

## 7 · Measured results (NVIDIA H100 80GB · vLLM 0.19.0 · bf16 · greedy · 256 tok × 10 prompts)

| Config | Throughput | Latency | Speedup |
|---|---:|---:|---:|
| baseline | 162.1 tok/s | 6.17 ms/tok | 1.00× |
| n-gram (K=5) | 139.8 tok/s | 7.16 ms/tok | **0.86× — slower!** |
| EAGLE-3 (K=3) | 334.6 tok/s | 2.99 ms/tok | **2.06×** |

Per-prompt spread: baseline is rock-steady (162 ± 0.2), while EAGLE-3 ranges **264–425
tok/s** depending on how predictable the text is — formulaic prose accelerates most,
varied code least.

Three sanity checks that make these numbers trustworthy:

1. **The baseline hits the roofline.** 3350 GB/s ÷ 14 GB ≈ 239 tok/s theoretical;
   162 measured = 68% — the usual gap once KV reads and scheduling are counted. A
   baseline far above the roofline would mean a broken benchmark.
2. **N-gram *should* lose here, and does.** Its proposals are free but weak on novel
   text (these prompts don't copy from themselves), so almost nothing is accepted —
   yet every step still pays verification bookkeeping. On a fast 7B target there's
   little latency to hide. N-gram wins on input-echoing workloads (summarization, RAG,
   edits), not on open-ended generation.
3. **EAGLE-3's variance is diagnostic.** Speedup tracks text predictability exactly as
   acceptance-rate theory says it must.

The production rule of thumb that falls out: **speculation pays when (a) the target is
large or the latency budget is tight, and (b) the proposer matches the workload.**
EAGLE-style heads are the strong default; draft models suit big targets with good small
siblings; n-gram only when outputs quote inputs; and batch-heavy serving may prefer no
speculation at all, since verification eats batch compute headroom.

## 8 · Takeaways

- Decode reads the whole model per token — intensity ≈ 1 FLOP/byte — so the GPU idles;
  the fix is more useful tokens per weight read, not more FLOP/s.
- Verify-in-bulk plus rejection sampling gives **mathematically identical outputs**; the
  draft can only cost speed, never quality (§3 proof).
- Measured on H100: EAGLE-3 doubles single-stream throughput (2.06×); n-gram loses 14%
  on this workload — *speculation is a workload decision, not a checkbox*.

### References

- Leviathan et al., *Fast Inference from Transformers via Speculative Decoding*, 2023 — [arXiv:2211.17192](https://arxiv.org/abs/2211.17192)
- Chen et al., *Accelerating LLM Decoding with Speculative Sampling*, 2023 — [arXiv:2302.01318](https://arxiv.org/abs/2302.01318)
- Kwon et al., *Efficient Memory Management for LLM Serving with PagedAttention* (vLLM), 2023 — [arXiv:2309.06180](https://arxiv.org/abs/2309.06180)
- Cai et al., *Medusa: Simple LLM Inference Acceleration with Multiple Decoding Heads*, 2024 — [arXiv:2401.10774](https://arxiv.org/abs/2401.10774)
- Li et al., *EAGLE: Speculative Sampling Requires Rethinking Feature Uncertainty*, 2024 — [arXiv:2401.15077](https://arxiv.org/abs/2401.15077)
- Li et al., *EAGLE-2: Faster Inference with Dynamic Draft Trees*, 2024 — [arXiv:2406.16858](https://arxiv.org/abs/2406.16858)
- Li et al., *EAGLE-3: Scaling up Inference Acceleration with Training-Time Test*, 2025 — [arXiv:2503.01840](https://arxiv.org/abs/2503.01840)
